# Automatic Differentiation

In [ ]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
# w 和 b 是优化参数，需要能够计算损失函数相对于这些变量的梯度，所以设置 requires_grad
w = torch.randn(5, 3, requires_grad=True)
# w.requires_grad_(True) # 也可以通过这种方式设置 requires_grad
b = torch.randn(3, requires_grad=True)
# z = wx + b
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
# 单层网络的损失函数 loss = z - y

## 梯度计算

In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

## 关闭梯度跟踪

In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

# 可以通过 torch.no_grad() 包围住无需计算梯度的部分
with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

# 另一种方式是使用detach函数来实现
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

一般需要关闭梯度跟踪的情况
- 一些参数在神经网络中被标记为冻结参数
- 当只做前向传播时，这样可以加快计算速度，因为对不跟踪梯度的张量计算会更高效

## 雅克比

In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

当使用相同的参数第二次调用backward时，梯度的值是不同的
发生这种情况是因为在进行反向传播时，PyTorch 会累积梯度，即将计算出的梯度值添加到计算图所有叶节点的 grad 属性中
如果想计算正确的梯度，需要先将 grad 属性清零
在现实训练中，优化器可以帮助做到这一点

## 示例

In [ ]:
a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)
Q = 3*a**3 - b**2

# 这里的Q是一个向量，所以需要显式传递梯度参数，表示 Q 与 Q 本身的梯度，即（1， 1）
external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)
# 等价地，也可以将 Q 聚合成标量并隐式调用
# Q.sum().backward()

# check if collected gradients are correct
print(9*a**2 == a.grad)
print(-2*b == b.grad)